# Angle drift diagnosis

Why does the ensemble-median posterior angle at window **2040** sit at 16.5-16.9 degrees for
*every* true angle, about +1.5 degrees above the prior angle median?

This notebook is a thin wrapper over `var_assim.plotting.angle_diagnostics` and
`analysis/scripts/angle_drift_diagnostics.py`. The script produces the full figure set and
report in one shot; the cells below are for poking at individual pieces interactively.

Scope: `ssp245` / `DEGpDEC 0.1` / `four` windowing / `Nens 1000`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from var_assim.plotting import angle_diagnostics as ad
from var_assim.plotting.presets import get_presets

presets, _ = get_presets()
plt.rcParams.update(presets)

THETAS = [5, 11, 15, 21, 30, 35]
SCREEN = 0.5

## Calibration and provenance

`B` and the `R_k` are not archived in the output files, so they are rebuilt from `config/`.
That is only valid if the runs used the *derived* observation weighting added in commit
85a2a91 -- `check_provenance` enforces it against both the current config and every file's
mtime.

In [3]:
args, Noise, Prior, Truth, Windowing = ad.build_calibration(15, model='pco2geowc_reg_noic')
paths = {th: ad.output_path(th) for th in THETAS}
ad.check_provenance(list(paths.values()), Noise)

windows = [str(w[1]) for w in Windowing.windows]
sigmas = ad.obs_sigmas(Noise)

print("obs weighting:", Noise.OBS_WEIGHTING)
print("obs sigmas   :", dict(zip(ad.OBS_VARS, np.round(sigmas, 4))))
print("windows      :", windows)
print("SAI ramp     :", args.tmin, "->", args.tmin + args.n_yrs_ramp)

obs weighting: derived
obs sigmas   : {'T1': 0.1, 'Q': 20.016, 'T_R1': 0.16, 'T_R2': 0.13}
windows      : ['2040', '2060', '2080', '2100']
SAI ramp     : 2025 -> 2075


## One window, one run

`load_window` reads only what the diagnostics need -- `data_hist` (7.5 GB per group) is never
touched, `controls_hist` only at `iter=0`, and `data_final` only for the four observable rows.

In [4]:
w = windows[0]
win = ad.load_window(paths[15], w, want_data_final=True)
keep = ad.screen_members(win["cost_hist"], SCREEN)
n_times = len(win["time"])

head = win["head"]
post = win["controls"][keep][:, head]
prior = win["controls_prior"][keep][:, head]

print(f"window {w}: n_times={n_times}  kept {keep.sum()} / {keep.size}")

window 2040: n_times=16  kept 999 / 1000


## Cost breakdown

`cost()` is a background quadratic plus four diagonal-weighted observation quadratics with no
cross terms, so the split is exact. The check that matters is `max_rel_residual`: the
recomputed total has to reproduce the archived `costs`.

In [5]:
Binv, prior_stds = ad.build_inv_covar_prior(Prior, Noise, n_times)
cb = ad.cost_breakdown(win, keep, Binv, sigmas)

print("max |dJ|/J vs archived costs:", f"{cb['max_rel_residual']:.2e}")
print()
for k in ("J_total", "J_prior", "J_obs", "J_truth"):
    print(f"  median {k:8s} = {np.median(cb[k]):10.3f}")
print()
for v in ad.OBS_VARS:
    print(f"  median J_obs_{v:5s} = {np.median(cb['J_obs_' + v]):8.3f}"
          f"   reduced chi2 = {np.median(cb['chi2red_' + v]):7.4f}")
print()
print("fraction of members with J(posterior) < J(truth):",
      float(np.mean(cb["J_total"] < cb["J_truth"])))

max |dJ|/J vs archived costs: 5.21e-08

  median J_total  =     29.350
  median J_prior  =     23.967
  median J_obs    =      5.239
  median J_truth  =     50.794

  median J_obs_T1    =    1.375   reduced chi2 =  0.1719
  median J_obs_Q     =    0.089   reduced chi2 =  0.0111
  median J_obs_T_R1  =    1.819   reduced chi2 =  0.2273
  median J_obs_T_R2  =    1.778   reduced chi2 =  0.2222

fraction of members with J(posterior) < J(truth): 1.0


## Angle offset attribution

Which control carries how much of the offset, in degrees. `residual` is the closure of the
first-order sum against the exact offset -- small means the per-parameter contributions can be
read as independent causes.

In [6]:
prior_med = ad.median_dict(prior)
post_med = ad.median_dict(post)
stds = {p: prior_stds[ad.CONTROL_HEAD.index(p)] for p in ad.ANGLE_PARAMS}

at = ad.attribute_angle_offset(prior_med, post_med, stds)

for p in ad.ANGLE_PARAMS:
    print(f"  {p:9s} shift {at['shift_sigma'][p]:+7.3f} sigma"
          f"   sens {at['sensitivity'][p]:+6.2f} deg/sigma"
          f"   -> {at['contrib_deg'][p]:+7.3f} deg")
print()
print(f"  first-order sum {at['offset_linear']:+.3f} deg")
print(f"  exact offset    {at['offset_exact']:+.3f} deg")
print(f"  residual        {at['residual']:+.3f} deg")
print()
for g, grp in ad.ANGLE_GROUPS.items():
    print(f"  exact, {g:5s} only: "
          f"{ad.substitute_angle(prior_med, post_med, grp) - ad.angle_at(prior_med):+.3f} deg")

  ALPHA_R1  shift  -0.128 sigma   sens  -2.04 deg/sigma   ->  +0.261 deg
  ALPHA_R2  shift  +0.012 sigma   sens  -0.49 deg/sigma   ->  -0.006 deg
  BETA_R1   shift  -0.087 sigma   sens  -2.59 deg/sigma   ->  +0.224 deg
  BETA_R2   shift  +0.069 sigma   sens  +6.04 deg/sigma   ->  +0.414 deg
  L         shift  +0.110 sigma   sens  +2.13 deg/sigma   ->  +0.234 deg
  EPS       shift  +0.014 sigma   sens  +0.60 deg/sigma   ->  +0.009 deg
  G         shift  +0.129 sigma   sens  +2.23 deg/sigma   ->  +0.287 deg

  first-order sum +1.424 deg
  exact offset    +1.460 deg
  residual        +0.036 deg

  exact, ALPHA only: +0.258 deg
  exact, BETA  only: +0.634 deg
  exact, LGE   only: +0.528 deg


## Is the shift the linear-Gaussian posterior response?

The assimilation is randomize-then-optimize: every member draws its own background from the
prior and minimizes `J` against the *same* observations. In the linear-Gaussian limit the
ensemble-mean posterior therefore sits at the prior centre plus `K (y - H x_cen)`.

Controls where prediction and measurement agree are responding correctly to this particular
realization of internal variability. Controls where they disagree are doing something else.

In [7]:
from var_assim.emis import EmissionsBaseline
from var_assim.models.pco2geowc_reg.dynamics import get_nonlin_path
from var_assim.models.pco2geowc_reg.obs import get_obs_from_dynamics
import logging

TMIN, TMAX = args.tmin, int(w)
e = EmissionsBaseline(logging.getLogger("nb"), args, TMIN, TMAX, geo=True,
                      Prior=Prior, Truth=Truth, T_START=TMIN,
                      T_END=TMIN + args.n_yrs_ramp, print_level=0)

x_cen = ad.prior_centre_from_ensemble(win)
H = ad.obs_jacobian(e, x_cen, TMIN, TMAX)
A, Rinv = ad.gauss_newton_system(H, Binv, sigmas, n_times)
pred_sd = np.sqrt(np.diag(np.linalg.inv(A)))

paths_cen, _ = get_nonlin_path(e, x_cen, TMIN, TMAX, 1.0)
innovation = (win["obs"] - get_obs_from_dynamics(paths_cen)).ravel()
dx_pred = ad.kalman_shift(H, A, Rinv, innovation)

dx_meas = np.median(post, axis=0) - np.median(prior, axis=0)
sd_meas = post.std(axis=0)
s = prior_stds[: ad.N_FIXED]

print(f"  {'control':10s} {'pred dx':>9s} {'meas dx':>9s} {'pred sd':>9s} {'meas sd':>9s}")
for j, n in enumerate(ad.CONTROL_HEAD):
    print(f"  {n:10s} {dx_pred[j]/s[j]:+9.3f} {dx_meas[j]/s[j]:+9.3f}"
          f" {pred_sd[j]/s[j]:9.3f} {sd_meas[j]/s[j]:9.3f}")

  control      pred dx   meas dx   pred sd   meas sd
  T1            +0.823    +0.793     0.721     0.716
  T2            +0.026    +0.068     0.353     0.413
  Q             -0.021    -0.039     0.556     0.324
  T_R1          +0.113    +0.115     0.433     0.388
  T_R2          +0.089    +0.096     0.444     0.413
  L             -0.023    +0.110     0.729     0.745
  G             +0.015    +0.129     0.869     0.904
  EPS           -0.003    +0.014     0.991     1.018
  C1            +0.060    -0.354     0.991     0.898
  C2            +0.007    -0.036     0.942     0.896
  F1_CO2        +0.020    +0.113     0.888     0.891
  ALPHA_R1      -0.201    -0.128     0.346     0.278
  ALPHA_R2      -0.001    +0.012     0.344     0.300
  BETA_R1       -0.073    -0.087     0.993     0.983
  BETA_R2       +0.046    +0.069     0.990     1.014


## The realization every run shares

All six runs use the same seeds, so the truth model errors in `controls_truth` are the same
realization in all of them. The observations are the noiseless truth path, so they *contain*
this realization -- and they are weighted more tightly than its own amplitude.

In [8]:
r = ad.realization_stats(win["controls_truth"], win["time"], n_times)
for b in ad.ERROR_BLOCKS:
    print(f"  {b}: mean {r[b]['mean']:+.4f}  std {r[b]['std']:.4f}"
          f"  trend {r[b]['trend']:+.5f}/yr  first-year {r[b]['first']:+.4f}")
print()
for name, so, si, f in ad.internal_variability_sigmas(Noise):
    print(f"  {name:5s} sigma_obs {so:.4g}  internal variability {si:.4g}"
          f"  -> {f:.2f}x too tight")
print()
g = ad.geo_information_split(e, win["time"], args.tmin + args.n_yrs_ramp)
print(f"  sum geo^2: ramp {g['ramp']:.1f} ({g['n_ramp']} yr)"
      f"   plateau {g['plateau']:.1f} ({g['n_plateau']} yr)")

  qAT: mean +0.0268  std 0.2556  trend +0.00352/yr  first-year +0.5053
  qR1: mean -0.0411  std 0.3102  trend -0.01821/yr  first-year +0.1315
  qR2: mean -0.0031  std 0.2814  trend +0.00284/yr  first-year +0.0567

  T1    sigma_obs 0.1  internal variability 0.27  -> 2.70x too tight
  T_R1  sigma_obs 0.16  internal variability 0.35  -> 2.19x too tight
  T_R2  sigma_obs 0.13  internal variability 0.3  -> 2.31x too tight

  sum geo^2: ramp 71.7 (16 yr)   plateau 0.0 (0 yr)


## Full run

Everything above, across all six runs and all four windows, with figures and a written report:

```
python analysis/scripts/angle_drift_diagnostics.py
```

Outputs land in `analysis/figs/results/` as dated PNGs plus `angle-diag-report.txt` and
`angle-diag-cost.csv`.